In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Project directory
PROJECT_DIR = Path.cwd().parent

# File paths
DATA_PATH = (
    PROJECT_DIR / "data" / "processed" / "forecasting_features.csv"
)

MODEL_DIR = PROJECT_DIR / "models"

print("Project directory:", PROJECT_DIR)
print("Dataset exists:", DATA_PATH.exists())
print("Models directory exists:", MODEL_DIR.exists())

Project directory: c:\Users\admin\OneDrive\Enterprise_Demand_Forecasting
Dataset exists: True
Models directory exists: True


In [2]:
# Load historical forecasting features
historical_df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

# Check dataset
print("Dataset loaded successfully!")
print("Total records:", len(historical_df))
print("Total products:", historical_df["item_id"].nunique())

print("\nLatest available date:")
print(historical_df["date"].max())

print("\nLatest record for each product:")
latest_records = (
    historical_df.sort_values("date")
    .groupby("item_id")
    .tail(1)
)

print(
    latest_records[
        ["item_id", "date", "sales", "sell_price"]
    ].to_string(index=False)
)

print("\nAvailable saved model files:")
for file in MODEL_DIR.glob("*"):
    if file.is_file():
        print(file.name)

Dataset loaded successfully!
Total records: 9089
Total products: 5

Latest available date:
2016-04-24 00:00:00

Latest record for each product:
    item_id       date  sales  sell_price
FOODS_3_586 2016-04-24     54        1.68
FOODS_3_090 2016-04-24     42        1.60
FOODS_3_120 2016-04-24     59        4.98
FOODS_3_252 2016-04-24     52        1.58
FOODS_3_714 2016-04-24     27        1.58

Available saved model files:
demand_forecasting_rf_v2.joblib
feature_columns_v2.joblib


In [3]:
# Load trained Random Forest V2
model_path = MODEL_DIR / "demand_forecasting_rf_v2.joblib"
features_path = MODEL_DIR / "feature_columns_v2.joblib"

forecast_model = joblib.load(model_path)
feature_columns = joblib.load(features_path)

print("Model loaded successfully!")
print("Model type:", type(forecast_model).__name__)
print("Number of features:", len(feature_columns))

print("\nRequired model features:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i}. {feature}")

print("\nModel feature count matches:",
      forecast_model.n_features_in_ == len(feature_columns))

Model loaded successfully!
Model type: RandomForestRegressor
Number of features: 23

Required model features:
1. day_of_week
2. day_of_month
3. week_of_year
4. month
5. year
6. is_weekend
7. lag_1
8. lag_7
9. lag_28
10. rolling_mean_7
11. rolling_mean_28
12. rolling_std_7
13. rolling_std_28
14. was_zero_yesterday
15. is_event
16. snap_CA
17. sell_price
18. price_available
19. product_FOODS_3_090
20. product_FOODS_3_120
21. product_FOODS_3_252
22. product_FOODS_3_586
23. product_FOODS_3_714

Model feature count matches: True


In [4]:
# Prepare historical sales for each product
sales_history = {}

for product in sorted(historical_df["item_id"].unique()):

    product_data = historical_df[
        historical_df["item_id"] == product
    ].sort_values("date")

    sales_history[product] = (
        product_data["sales"].astype(float).tolist()
    )

# Verify historical sequences
print("Historical sales sequences prepared!\n")

for product, history in sales_history.items():
    print(
        f"{product}: "
        f"{len(history)} days | "
        f"Last 7 sales: {history[-7:]}"
    )

Historical sales sequences prepared!

FOODS_3_090: 1885 days | Last 7 sales: [30.0, 45.0, 29.0, 53.0, 87.0, 95.0, 42.0]
FOODS_3_120: 1549 days | Last 7 sales: [6.0, 43.0, 48.0, 47.0, 39.0, 59.0, 59.0]
FOODS_3_252: 1885 days | Last 7 sales: [21.0, 46.0, 24.0, 35.0, 33.0, 49.0, 52.0]
FOODS_3_586: 1885 days | Last 7 sales: [30.0, 40.0, 31.0, 45.0, 28.0, 40.0, 54.0]
FOODS_3_714: 1885 days | Last 7 sales: [15.0, 6.0, 10.0, 12.0, 18.0, 22.0, 27.0]


In [5]:
import pandas as pd

# Define the next 7 forecast dates
future_dates = pd.date_range(
    start="2016-04-25",
    periods=7,
    freq="D"
)

# Load the original M5 calendar
calendar_path = PROJECT_DIR / "data" / "raw" / "calendar.csv"

future_calendar = pd.read_csv(calendar_path)
future_calendar["date"] = pd.to_datetime(
    future_calendar["date"]
)

# Check available calendar records
future_calendar = future_calendar[
    future_calendar["date"].isin(future_dates)
].copy()

print("Future calendar records found:", len(future_calendar))

print("\nFuture calendar information:")
print(
    future_calendar[
        ["date", "wm_yr_wk", "event_name_1", "snap_CA"]
    ].to_string(index=False)
)

Future calendar records found: 7

Future calendar information:
      date  wm_yr_wk   event_name_1  snap_CA
2016-04-25     11613            NaN        0
2016-04-26     11613            NaN        0
2016-04-27     11613            NaN        0
2016-04-28     11613            NaN        0
2016-04-29     11613            NaN        0
2016-04-30     11614     Pesach End        0
2016-05-01     11614 OrthodoxEaster        1


In [6]:
# Load original selling prices
prices_path = PROJECT_DIR / "data" / "raw" / "sell_prices.csv"

prices_df = pd.read_csv(prices_path)

# Get the future Walmart weeks
future_weeks = sorted(
    future_calendar["wm_yr_wk"].unique()
)

# Get our five products
selected_products = sorted(sales_history.keys())

# Filter prices for CA_1 store and future weeks
future_prices = prices_df[
    (prices_df["store_id"] == "CA_1")
    & (prices_df["item_id"].isin(selected_products))
    & (prices_df["wm_yr_wk"].isin(future_weeks))
].copy()

print("Future Walmart weeks:", future_weeks)
print("Expected price records:", len(selected_products) * len(future_weeks))
print("Available price records:", len(future_prices))

print("\nFuture selling prices:")
print(
    future_prices[
        ["item_id", "wm_yr_wk", "sell_price"]
    ].sort_values(
        ["item_id", "wm_yr_wk"]
    ).to_string(index=False)
)

print("\nMissing prices:", future_prices["sell_price"].isna().sum())

Future Walmart weeks: [np.int64(11613), np.int64(11614)]
Expected price records: 10
Available price records: 10

Future selling prices:
    item_id  wm_yr_wk  sell_price
FOODS_3_090     11613        1.60
FOODS_3_090     11614        1.60
FOODS_3_120     11613        4.98
FOODS_3_120     11614        4.98
FOODS_3_252     11613        1.58
FOODS_3_252     11614        1.58
FOODS_3_586     11613        1.68
FOODS_3_586     11614        1.68
FOODS_3_714     11613        1.58
FOODS_3_714     11614        1.58

Missing prices: 0


In [8]:
# Prepare first future prediction
product = "FOODS_3_090"
forecast_date = pd.Timestamp("2016-04-25")

# Historical sales up to April 24
history = sales_history[product]

# Get future calendar details
calendar_row = future_calendar[
    future_calendar["date"] == forecast_date
].iloc[0]

# Get future selling price
price_row = future_prices[
    (future_prices["item_id"] == product)
    & (future_prices["wm_yr_wk"] == calendar_row["wm_yr_wk"])
].iloc[0]

# Construct feature values
feature_values = {
    "day_of_week": forecast_date.day_of_week,
    "day_of_month": forecast_date.day,
    "week_of_year": forecast_date.isocalendar().week,
    "month": forecast_date.month,
    "year": forecast_date.year,
    "is_weekend": int(forecast_date.dayofweek >= 5),

    "lag_1": history[-1],
    "lag_7": history[-7],
    "lag_28": history[-28],

    "rolling_mean_7": np.mean(history[-7:]),
    "rolling_mean_28": np.mean(history[-28:]),
    "rolling_std_7": pd.Series(history[-7:]).std(),
    "rolling_std_28": pd.Series(history[-28:]).std(),

    "was_zero_yesterday": int(history[-1] == 0),

    "is_event": int(pd.notna(calendar_row["event_name_1"])),
    "snap_CA": int(calendar_row["snap_CA"]),

    "sell_price": float(price_row["sell_price"]),
    "price_available": 1
}

# One-hot encode product identity
for item in selected_products:
    feature_values[f"product_{item}"] = int(item == product)

# Match the exact training feature order
future_X = pd.DataFrame(
    [feature_values]
)[feature_columns]

print("Product:", product)
print("Forecast date:", forecast_date.date())
print("Features prepared:", future_X.shape[1])
print("Missing values:", future_X.isna().sum().sum())

display(future_X.T)

Product: FOODS_3_090
Forecast date: 2016-04-25
Features prepared: 23
Missing values: 0


C:\Users\admin\AppData\Local\Temp\ipykernel_5828\645526993.py:26: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  "is_weekend": int(forecast_date.dayofweek >= 5),


,0
day_of_week,0.000000
day_of_month,25.000000
week_of_year,17.000000
month,4.000000
year,2016.000000
is_weekend,0.000000
lag_1,42.000000
lag_7,30.000000
lag_28,25.000000
rolling_mean_7,54.428571


In [9]:
# Compare calculated features with original training data
product_df = historical_df[
    historical_df["item_id"] == "FOODS_3_090"
].sort_values("date")

previous_sales = product_df["sales"].iloc[:-1]
latest_row = product_df.iloc[-1]

checks = {
    "lag_1": previous_sales.iloc[-1],
    "lag_7": previous_sales.iloc[-7],
    "lag_28": previous_sales.iloc[-28],
    "rolling_mean_7": previous_sales.iloc[-7:].mean(),
    "rolling_mean_28": previous_sales.iloc[-28:].mean(),
    "rolling_std_7": previous_sales.iloc[-7:].std(),
    "rolling_std_28": previous_sales.iloc[-28:].std()
}

comparison = pd.DataFrame({
    "Original": [latest_row[f] for f in checks],
    "Calculated": list(checks.values())
}, index=list(checks.keys()))

comparison["Difference"] = (
    comparison["Original"] - comparison["Calculated"]
)

display(comparison)

print(
    "All features match:",
    np.allclose(
        comparison["Original"],
        comparison["Calculated"],
        atol=1e-5
    )
)

,Original,Calculated,Difference
lag_1,95.000000,95.000000,0.000000e+00
lag_7,83.000000,83.000000,0.000000e+00
lag_28,112.000000,112.000000,0.000000e+00
rolling_mean_7,60.285714,60.285714,0.000000e+00
rolling_mean_28,53.321429,53.321429,0.000000e+00
rolling_std_7,27.741150,27.741150,3.552714e-14
rolling_std_28,24.546333,24.546333,-3.552714e-15


All features match: True


In [10]:
# Generate first future sales prediction
prediction = forecast_model.predict(future_X)

# Sales cannot be negative
predicted_sales = max(0, float(prediction[0]))

print("FIRST FUTURE DEMAND FORECAST")
print("----------------------------")
print("Product:", product)
print("Forecast Date:", forecast_date.date())
print("Predicted Sales:", round(predicted_sales, 2), "units")
print("Rounded Demand:", round(predicted_sales), "units")

FIRST FUTURE DEMAND FORECAST
----------------------------
Product: FOODS_3_090
Forecast Date: 2016-04-25
Predicted Sales: 40.62 units
Rounded Demand: 41 units


In [12]:
def forecast_next_days(product, days=7):

    # Copy history to avoid modifying original data
    history = sales_history[product].copy()

    forecasts = []

    for date in future_dates[:days]:

        calendar_row = future_calendar[
            future_calendar["date"] == date
        ].iloc[0]

        price_row = future_prices[
            (future_prices["item_id"] == product)
            & (
                future_prices["wm_yr_wk"]
                == calendar_row["wm_yr_wk"]
            )
        ].iloc[0]

        features = {
            "day_of_week": date.day_of_week,
            "day_of_month": date.day,
            "week_of_year": date.isocalendar().week,
            "month": date.month,
            "year": date.year,
            "is_weekend": int(date.day_of_week >= 5),

            "lag_1": history[-1],
            "lag_7": history[-7],
            "lag_28": history[-28],

            "rolling_mean_7": np.mean(history[-7:]),
            "rolling_mean_28": np.mean(history[-28:]),
            "rolling_std_7": pd.Series(history[-7:]).std(),
            "rolling_std_28": pd.Series(history[-28:]).std(),

            "was_zero_yesterday": int(history[-1] == 0),

            "is_event": int(
                pd.notna(calendar_row["event_name_1"])
            ),
            "snap_CA": int(calendar_row["snap_CA"]),

            "sell_price": float(price_row["sell_price"]),
            "price_available": 1
        }

        # Product one-hot encoding
        for item in selected_products:
            features[f"product_{item}"] = int(item == product)

        X = pd.DataFrame([features])[feature_columns]

        prediction = max(
            0,
            float(forecast_model.predict(X)[0])
        )

        forecasts.append({
            "item_id": product,
            "date": date,
            "predicted_sales": prediction
        })

        # Use prediction for subsequent days
        history.append(prediction)

    return pd.DataFrame(forecasts)

In [13]:
print("Function created:", callable(forecast_next_days))

Function created: True


In [14]:
product_forecast = forecast_next_days(
    "FOODS_3_090",
    days=7
)

print("7-Day Forecast Results:")
print(product_forecast.to_string(index=False))

7-Day Forecast Results:
    item_id       date  predicted_sales
FOODS_3_090 2016-04-25        40.615530
FOODS_3_090 2016-04-26        42.416609
FOODS_3_090 2016-04-27        42.766473
FOODS_3_090 2016-04-28        45.165820
FOODS_3_090 2016-04-29        59.997045
FOODS_3_090 2016-04-30        68.031649
FOODS_3_090 2016-05-01        73.380788


In [15]:
# Generate 7-day forecasts for all five products
all_forecasts = []

for item in selected_products:

    forecast_df = forecast_next_days(
        product=item,
        days=7
    )

    all_forecasts.append(forecast_df)

# Combine forecasts into one DataFrame
future_forecast_df = pd.concat(
    all_forecasts,
    ignore_index=True
)

# Display results
print("Total predictions:", len(future_forecast_df))
print("Products:", future_forecast_df["item_id"].nunique())
print("Forecast days:", future_forecast_df["date"].nunique())

print("\n7-Day Forecast Summary:")

summary = (
    future_forecast_df
    .groupby("item_id")["predicted_sales"]
    .agg(["sum", "mean", "max"])
    .round(2)
)

display(summary)

Total predictions: 35
Products: 5
Forecast days: 7

7-Day Forecast Summary:


,sum,mean,max
item_id,,,
FOODS_3_090,372.37,53.20,73.38
FOODS_3_120,302.06,43.15,58.98
FOODS_3_252,254.16,36.31,55.80
FOODS_3_586,267.50,38.21,56.00
FOODS_3_714,132.24,18.89,29.79


In [16]:
# Save future demand forecasts
output_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "future_forecasts.csv"
)

future_forecast_df.to_csv(
    output_path,
    index=False
)

print("Forecast file saved successfully!")
print("File location:", output_path)
print("Total rows:", len(future_forecast_df))

# Verify saved file
saved_forecasts = pd.read_csv(output_path)

print("Saved file verified:", len(saved_forecasts) == 35)
print("\nPreview:")
print(saved_forecasts.head().to_string(index=False))

Forecast file saved successfully!
File location: c:\Users\admin\OneDrive\Enterprise_Demand_Forecasting\data\processed\future_forecasts.csv
Total rows: 35
Saved file verified: True

Preview:
    item_id       date  predicted_sales
FOODS_3_090 2016-04-25        40.615530
FOODS_3_090 2016-04-26        42.416609
FOODS_3_090 2016-04-27        42.766473
FOODS_3_090 2016-04-28        45.165820
FOODS_3_090 2016-04-29        59.997045


In [17]:
# Select historical period for backtesting
backtest_start = pd.Timestamp("2016-04-01")
backtest_end = pd.Timestamp("2016-04-07")

backtest_actuals = historical_df[
    (historical_df["date"] >= backtest_start)
    & (historical_df["date"] <= backtest_end)
].copy()

print("Backtesting period:")
print(backtest_start.date(), "to", backtest_end.date())

print("\nTotal actual records:", len(backtest_actuals))
print("Products:", backtest_actuals["item_id"].nunique())
print("Days:", backtest_actuals["date"].nunique())

print("\nActual sales:")
display(
    backtest_actuals[
        ["item_id", "date", "sales"]
    ].sort_values(["item_id", "date"])
)

Backtesting period:
2016-04-01 to 2016-04-07

Total actual records: 35
Products: 5
Days: 7

Actual sales:


,item_id,date,sales
8969,FOODS_3_090,2016-04-01,72
8974,FOODS_3_090,2016-04-02,58
8979,FOODS_3_090,2016-04-03,54
8984,FOODS_3_090,2016-04-04,42
8989,FOODS_3_090,2016-04-05,29
8994,FOODS_3_090,2016-04-06,53
8999,FOODS_3_090,2016-04-07,39
8970,FOODS_3_120,2016-04-01,44
8975,FOODS_3_120,2016-04-02,109
8980,FOODS_3_120,2016-04-03,126


In [18]:
# Historical cutoff for backtesting
backtest_cutoff = pd.Timestamp("2016-03-31")

# Store sales history before the backtest period
backtest_history = {}

for item in selected_products:

    product_data = historical_df[
        (historical_df["item_id"] == item)
        & (historical_df["date"] <= backtest_cutoff)
    ].sort_values("date")

    backtest_history[item] = (
        product_data["sales"].astype(float).tolist()
    )

print("Backtest history prepared!\n")

for item, history in backtest_history.items():
    print(
        f"{item}: {len(history)} historical days | "
        f"Last sale: {history[-1]}"
    )

print("\nBacktest cutoff:", backtest_cutoff.date())

Backtest history prepared!

FOODS_3_090: 1861 historical days | Last sale: 21.0
FOODS_3_120: 1525 historical days | Last sale: 29.0
FOODS_3_252: 1861 historical days | Last sale: 26.0
FOODS_3_586: 1861 historical days | Last sale: 28.0
FOODS_3_714: 1861 historical days | Last sale: 19.0

Backtest cutoff: 2016-03-31


In [19]:
# Verify daily continuity for each product
continuity_results = []

for item in selected_products:

    product_data = historical_df[
        (historical_df["item_id"] == item)
        & (historical_df["date"] <= backtest_cutoff)
    ].sort_values("date")

    expected_dates = pd.date_range(
        start=product_data["date"].min(),
        end=backtest_cutoff,
        freq="D"
    )

    missing_dates = expected_dates.difference(
        product_data["date"]
    )

    continuity_results.append({
        "item_id": item,
        "first_date": product_data["date"].min().date(),
        "records": len(product_data),
        "missing_days": len(missing_dates)
    })

continuity_df = pd.DataFrame(continuity_results)

display(continuity_df)

print(
    "All histories continuous:",
    (continuity_df["missing_days"] == 0).all()
)

,item_id,first_date,records,missing_days
0,FOODS_3_090,2011-02-26,1861,0
1,FOODS_3_120,2012-01-28,1525,0
2,FOODS_3_252,2011-02-26,1861,0
3,FOODS_3_586,2011-02-26,1861,0
4,FOODS_3_714,2011-02-26,1861,0


All histories continuous: True


In [20]:
# Prepare 7-day backtesting dates
backtest_dates = pd.date_range(
    start=backtest_start,
    end=backtest_end,
    freq="D"
)

# Load original calendar for backtesting
backtest_calendar = pd.read_csv(calendar_path)

backtest_calendar["date"] = pd.to_datetime(
    backtest_calendar["date"]
)

backtest_calendar = backtest_calendar[
    backtest_calendar["date"].isin(backtest_dates)
].copy()

# Get Walmart weeks
backtest_weeks = backtest_calendar[
    "wm_yr_wk"
].unique()

# Retrieve selling prices
backtest_prices = prices_df[
    (prices_df["store_id"] == "CA_1")
    & (prices_df["item_id"].isin(selected_products))
    & (prices_df["wm_yr_wk"].isin(backtest_weeks))
].copy()

print("Backtest calendar records:", len(backtest_calendar))
print("Backtest price records:", len(backtest_prices))

print("\nBacktesting calendar:")
print(
    backtest_calendar[
        ["date", "wm_yr_wk", "event_name_1", "snap_CA"]
    ].to_string(index=False)
)

print("\nMissing price values:",
      backtest_prices["sell_price"].isna().sum())

Backtest calendar records: 7
Backtest price records: 10

Backtesting calendar:
      date  wm_yr_wk event_name_1  snap_CA
2016-04-01     11609          NaN        1
2016-04-02     11610          NaN        1
2016-04-03     11610          NaN        1
2016-04-04     11610          NaN        1
2016-04-05     11610          NaN        1
2016-04-06     11610          NaN        1
2016-04-07     11610          NaN        1

Missing price values: 0


In [21]:
# Recursive backtesting function
def forecast_backtest(product, days=7):

    history = backtest_history[product].copy()
    predictions = []

    for date in backtest_dates[:days]:

        calendar_row = backtest_calendar[
            backtest_calendar["date"] == date
        ].iloc[0]

        price_row = backtest_prices[
            (backtest_prices["item_id"] == product)
            & (
                backtest_prices["wm_yr_wk"]
                == calendar_row["wm_yr_wk"]
            )
        ].iloc[0]

        features = {
            "day_of_week": date.day_of_week,
            "day_of_month": date.day,
            "week_of_year": date.isocalendar().week,
            "month": date.month,
            "year": date.year,
            "is_weekend": int(date.day_of_week >= 5),

            "lag_1": history[-1],
            "lag_7": history[-7],
            "lag_28": history[-28],

            "rolling_mean_7": np.mean(history[-7:]),
            "rolling_mean_28": np.mean(history[-28:]),
            "rolling_std_7": pd.Series(history[-7:]).std(),
            "rolling_std_28": pd.Series(history[-28:]).std(),

            "was_zero_yesterday": int(history[-1] == 0),
            "is_event": int(
                pd.notna(calendar_row["event_name_1"])
            ),
            "snap_CA": int(calendar_row["snap_CA"]),
            "sell_price": float(price_row["sell_price"]),
            "price_available": 1
        }

        for item in selected_products:
            features[f"product_{item}"] = int(item == product)

        X = pd.DataFrame([features])[feature_columns]

        predicted_sales = max(
            0,
            float(forecast_model.predict(X)[0])
        )

        predictions.append({
            "item_id": product,
            "date": date,
            "predicted_sales": predicted_sales
        })

        # Append prediction, never actual future sales
        history.append(predicted_sales)

    return pd.DataFrame(predictions)


# Test with one product
backtest_prediction = forecast_backtest(
    "FOODS_3_090",
    days=7
)

print("Recursive backtest predictions:")
display(backtest_prediction)

print(
    "Total predicted demand:",
    round(backtest_prediction["predicted_sales"].sum(), 2)
)

Recursive backtest predictions:


,item_id,date,predicted_sales
0,FOODS_3_090,2016-04-01,42.393802
1,FOODS_3_090,2016-04-02,61.793650
2,FOODS_3_090,2016-04-03,83.509936
3,FOODS_3_090,2016-04-04,52.534371
4,FOODS_3_090,2016-04-05,44.808831
5,FOODS_3_090,2016-04-06,41.006104
6,FOODS_3_090,2016-04-07,42.666623


Total predicted demand: 368.71


In [22]:
# Get actual sales for the same product and dates
actual_sales = backtest_actuals[
    backtest_actuals["item_id"] == "FOODS_3_090"
][["date", "sales"]].copy()

# Combine actual and predicted sales
comparison_df = backtest_prediction.merge(
    actual_sales,
    on="date",
    how="inner"
)

# Calculate absolute errors
comparison_df["absolute_error"] = (
    comparison_df["sales"]
    - comparison_df["predicted_sales"]
).abs()

# Calculate evaluation metrics
mae = comparison_df["absolute_error"].mean()

rmse = np.sqrt(
    np.mean(
        (
            comparison_df["sales"]
            - comparison_df["predicted_sales"]
        ) ** 2
    )
)

print("7-Day Recursive Forecast Evaluation")
print("-----------------------------------")
print("Product: FOODS_3_090")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

print("\nActual vs Predicted Sales:")
display(
    comparison_df[
        ["date", "sales", "predicted_sales", "absolute_error"]
    ]
)

7-Day Recursive Forecast Evaluation
-----------------------------------
Product: FOODS_3_090
MAE: 14.99
RMSE: 18.05

Actual vs Predicted Sales:


,date,sales,predicted_sales,absolute_error
0,2016-04-01,72,42.393802,29.606198
1,2016-04-02,58,61.793650,3.793650
2,2016-04-03,54,83.509936,29.509936
3,2016-04-04,42,52.534371,10.534371
4,2016-04-05,29,44.808831,15.808831
5,2016-04-06,53,41.006104,11.993896
6,2016-04-07,39,42.666623,3.666623


In [23]:
# Inspect historical dataset date ranges
print("Complete dataset:")
print("Start:", historical_df["date"].min().date())
print("End:", historical_df["date"].max().date())

# Inspect the original chronological split
dates = sorted(historical_df["date"].unique())

n_dates = len(dates)

print("\nTotal unique dates:", n_dates)

print("\nOriginal project test period:")
print("Start: 2015-07-17")
print("End: 2016-04-24")

print("\nPotential earlier backtesting window:")
print("Start: 2015-07-17")
print("End: 2015-07-23")

# Check actual sales availability
candidate_start = pd.Timestamp("2015-07-17")
candidate_end = pd.Timestamp("2015-07-23")

candidate_data = historical_df[
    historical_df["date"].between(
        candidate_start,
        candidate_end
    )
]

print("\nAvailable records:", len(candidate_data))
print("Products:", candidate_data["item_id"].nunique())
print("Days:", candidate_data["date"].nunique())

Complete dataset:
Start: 2011-02-26
End: 2016-04-24

Total unique dates: 1885

Original project test period:
Start: 2015-07-17
End: 2016-04-24

Potential earlier backtesting window:
Start: 2015-07-17
End: 2015-07-23

Available records: 35
Products: 5
Days: 7


In [24]:
# Check saved model information and training artifacts
print("Saved model files:\n")

for file in MODEL_DIR.iterdir():
    if file.is_file():
        print(file.name)

print("\nModel details:")
print("Model type:", type(forecast_model).__name__)
print("Number of features:", forecast_model.n_features_in_)
print("Number of trees:", forecast_model.n_estimators)

print("\nTraining-related files:\n")

for folder in [
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR / "notebooks"
]:
    print(f"\n{folder.name}/")

    for file in folder.iterdir():
        if file.is_file():
            print(" -", file.name)

Saved model files:

demand_forecasting_rf_v2.joblib
feature_columns_v2.joblib

Model details:
Model type: RandomForestRegressor
Number of features: 23
Number of trees: 100

Training-related files:


processed/
 - forecasting_features.csv
 - future_forecasts.csv
 - test_predictions.csv

notebooks/
 - 01_data_exploration.ipynb
 - 02_future_forecasting.ipynb


In [25]:
import json

notebook_path = (
    PROJECT_DIR
    / "notebooks"
    / "01_data_exploration.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "RandomForestRegressor",
    "train_test_split",
    "fit(",
    "joblib.dump",
    "train_df",
    "val_df",
    "test_df"
]

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(word in source for word in keywords):
        print(f"\n{'=' * 45}")
        print(f"CELL {i}")
        print(source[:2500])


CELL 31
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Create the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict validation sales
val_predictions = rf_model.predict(X_val)

# Evaluate predictions
mae = mean_absolute_error(y_val, val_predictions)
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))

print("Random Forest Model Results")
print("---------------------------")
print("Validation MAE:", round(mae, 2))
print("Validation RMSE:", round(rmse, 2))

CELL 39
# Train Random Forest with product identity features
rf_model_v2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_model_v2.fit(X_train_v2, y_train)

# Predict validation sales
val_pred

In [26]:
# Find the original train, validation and test split

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(
        keyword in source
        for keyword in [
            "train_data =",
            "val_data =",
            "test_data =",
            "train_end",
            "val_end",
            "train_size",
            "val_size"
        ]
    ):
        print("\n" + "=" * 45)
        print("CELL", i)
        print(source[:3500])


CELL 29
# Get all unique dates in chronological order
unique_dates = sorted(model_data["date"].unique())

# Calculate split boundaries
train_end = int(len(unique_dates) * 0.70)
val_end = int(len(unique_dates) * 0.85)

# Split using dates, not random rows
train_dates = unique_dates[:train_end]
val_dates = unique_dates[train_end:val_end]
test_dates = unique_dates[val_end:]

train_data = model_data[
    model_data["date"].isin(train_dates)
].copy()

val_data = model_data[
    model_data["date"].isin(val_dates)
].copy()

test_data = model_data[
    model_data["date"].isin(test_dates)
].copy()

print("Training:", train_data.shape)
print("Validation:", val_data.shape)
print("Testing:", test_data.shape)

print("\nTraining dates:",
      train_data["date"].min(), "to",
      train_data["date"].max())

print("Validation dates:",
      val_data["date"].min(), "to",
      val_data["date"].max())

print("Testing dates:",
      test_data["date"].min(), "to",
      test_data["date"].max())


In [27]:
# Reconstruct chronological split boundaries
unique_dates = sorted(historical_df["date"].unique())

train_end = int(len(unique_dates) * 0.70)
val_end = int(len(unique_dates) * 0.85)

print("Training ends:", unique_dates[train_end - 1].date())
print("Validation starts:", unique_dates[train_end].date())
print("Validation ends:", unique_dates[val_end - 1].date())
print("Testing starts:", unique_dates[val_end].date())
print("Testing ends:", unique_dates[-1].date())

# Verify proposed backtesting dates
candidate_start = pd.Timestamp("2015-07-17")
candidate_end = pd.Timestamp("2015-07-23")

print("\nCandidate backtest is inside test period:",
      candidate_start >= unique_dates[val_end]
      and candidate_end <= unique_dates[-1])

Training ends: 2014-10-06
Validation starts: 2014-10-07
Validation ends: 2015-07-16
Testing starts: 2015-07-17
Testing ends: 2016-04-24

Candidate backtest is inside test period: True


In [28]:
# Define held-out backtesting period
heldout_start = pd.Timestamp("2015-07-17")
heldout_end = pd.Timestamp("2015-07-23")

heldout_dates = pd.date_range(
    start=heldout_start,
    end=heldout_end,
    freq="D"
)

# Prepare sales history available before forecast start
heldout_history = {}

for item in selected_products:

    product_data = historical_df[
        (historical_df["item_id"] == item)
        & (historical_df["date"] < heldout_start)
    ].sort_values("date")

    heldout_history[item] = (
        product_data["sales"].astype(float).tolist()
    )

# Verify historical data cutoff
print("Held-out backtest starts:", heldout_start.date())
print("Last permitted actual sales date:",
      (heldout_start - pd.Timedelta(days=1)).date())

print("\nHistorical observations:")

for item, history in heldout_history.items():
    print(
        f"{item}: {len(history)} days | "
        f"Last sale: {history[-1]}"
    )

print("\nAll products have at least 28 observations:",
      all(len(h) >= 28 for h in heldout_history.values()))

Held-out backtest starts: 2015-07-17
Last permitted actual sales date: 2015-07-16

Historical observations:
FOODS_3_090: 1602 days | Last sale: 62.0
FOODS_3_120: 1266 days | Last sale: 18.0
FOODS_3_252: 1602 days | Last sale: 35.0
FOODS_3_586: 1602 days | Last sale: 36.0
FOODS_3_714: 1602 days | Last sale: 30.0

All products have at least 28 observations: True


In [29]:
# Prepare held-out calendar
heldout_calendar = pd.read_csv(calendar_path)

heldout_calendar["date"] = pd.to_datetime(
    heldout_calendar["date"]
)

heldout_calendar = heldout_calendar[
    heldout_calendar["date"].isin(heldout_dates)
].copy()

# Identify Walmart weeks
heldout_weeks = heldout_calendar["wm_yr_wk"].unique()

# Prepare held-out selling prices
heldout_prices = prices_df[
    (prices_df["store_id"] == "CA_1")
    & (prices_df["item_id"].isin(selected_products))
    & (prices_df["wm_yr_wk"].isin(heldout_weeks))
].copy()

# Check data completeness
print("Held-out calendar records:", len(heldout_calendar))
print("Held-out price records:", len(heldout_prices))

print("\nCalendar details:")
print(
    heldout_calendar[
        ["date", "wm_yr_wk", "event_name_1", "snap_CA"]
    ].to_string(index=False)
)

print("\nMissing price values:",
      heldout_prices["sell_price"].isna().sum())

print("\nExpected price records:",
      len(selected_products) * len(heldout_weeks))

Held-out calendar records: 7
Held-out price records: 10

Calendar details:
      date  wm_yr_wk event_name_1  snap_CA
2015-07-17     11524          NaN        0
2015-07-18     11525  Eid al-Fitr        0
2015-07-19     11525          NaN        0
2015-07-20     11525          NaN        0
2015-07-21     11525          NaN        0
2015-07-22     11525          NaN        0
2015-07-23     11525          NaN        0

Missing price values: 0

Expected price records: 10


In [30]:
# Preserve the previous backtest inputs
old_history = backtest_history
old_dates = backtest_dates
old_calendar = backtest_calendar
old_prices = backtest_prices

try:
    # Supply held-out backtesting inputs
    backtest_history = heldout_history
    backtest_dates = heldout_dates
    backtest_calendar = heldout_calendar
    backtest_prices = heldout_prices

    # Generate predictions for all five products
    heldout_predictions = pd.concat(
        [
            forecast_backtest(item, days=7)
            for item in selected_products
        ],
        ignore_index=True
    )

finally:
    # Restore original inputs
    backtest_history = old_history
    backtest_dates = old_dates
    backtest_calendar = old_calendar
    backtest_prices = old_prices

# Verify predictions
print("Held-out predictions generated!")
print("Total predictions:", len(heldout_predictions))
print("Products:", heldout_predictions["item_id"].nunique())
print("Days:", heldout_predictions["date"].nunique())

print("\nPrediction preview:")
display(heldout_predictions.head(10))

Held-out predictions generated!
Total predictions: 35
Products: 5
Days: 7

Prediction preview:


,item_id,date,predicted_sales
0,FOODS_3_090,2015-07-17,82.327760
1,FOODS_3_090,2015-07-18,92.180783
2,FOODS_3_090,2015-07-19,91.678964
3,FOODS_3_090,2015-07-20,55.345552
4,FOODS_3_090,2015-07-21,53.262669
5,FOODS_3_090,2015-07-22,52.727912
6,FOODS_3_090,2015-07-23,55.273953
7,FOODS_3_120,2015-07-17,40.388753
8,FOODS_3_120,2015-07-18,55.361086
9,FOODS_3_120,2015-07-19,60.290023


In [31]:
# Get actual sales for the held-out period
heldout_actuals = historical_df[
    historical_df["date"].between(
        heldout_start,
        heldout_end
    )
][["item_id", "date", "sales"]].copy()

# Merge predictions with actual sales
heldout_results = heldout_predictions.merge(
    heldout_actuals,
    on=["item_id", "date"],
    how="inner",
    validate="one_to_one"
)

# Verify all predictions were matched
assert len(heldout_results) == 35

# Calculate prediction errors
heldout_results["absolute_error"] = (
    heldout_results["sales"]
    - heldout_results["predicted_sales"]
).abs()

heldout_results["squared_error"] = (
    heldout_results["sales"]
    - heldout_results["predicted_sales"]
) ** 2

# Overall evaluation metrics
overall_mae = heldout_results["absolute_error"].mean()

overall_rmse = np.sqrt(
    heldout_results["squared_error"].mean()
)

print("7-DAY RECURSIVE FORECAST EVALUATION")
print("-----------------------------------")
print("Period: July 17–23, 2015")
print("Total predictions:", len(heldout_results))
print("MAE:", round(overall_mae, 2))
print("RMSE:", round(overall_rmse, 2))

# Product-level performance
product_metrics = heldout_results.groupby("item_id").agg(
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean")
)

product_metrics["RMSE"] = np.sqrt(
    product_metrics["MSE"]
)

product_metrics = product_metrics[
    ["MAE", "RMSE"]
].round(2)

print("\nProduct-Level Forecast Accuracy:")
display(product_metrics)

7-DAY RECURSIVE FORECAST EVALUATION
-----------------------------------
Period: July 17–23, 2015
Total predictions: 35
MAE: 12.69
RMSE: 20.91

Product-Level Forecast Accuracy:


,MAE,RMSE
item_id,,
FOODS_3_090,21.09,33.38
FOODS_3_120,24.78,29.62
FOODS_3_252,5.33,7.88
FOODS_3_586,6.87,8.92
FOODS_3_714,5.39,7.31


In [32]:
# Seasonal naive baseline: repeat the previous week's sales
baseline_predictions = []

for item in selected_products:

    history = heldout_history[item]

    # Last 7 observed sales before the forecast starts
    previous_week = history[-7:]

    for i, date in enumerate(heldout_dates):

        baseline_predictions.append({
            "item_id": item,
            "date": date,
            "baseline_sales": previous_week[i]
        })

baseline_df = pd.DataFrame(baseline_predictions)

# Join baseline with model predictions and actual sales
baseline_comparison = heldout_results.merge(
    baseline_df,
    on=["item_id", "date"],
    how="inner",
    validate="one_to_one"
)

# Calculate baseline errors
baseline_comparison["baseline_absolute_error"] = (
    baseline_comparison["sales"]
    - baseline_comparison["baseline_sales"]
).abs()

baseline_comparison["baseline_squared_error"] = (
    baseline_comparison["sales"]
    - baseline_comparison["baseline_sales"]
) ** 2

baseline_mae = baseline_comparison[
    "baseline_absolute_error"
].mean()

baseline_rmse = np.sqrt(
    baseline_comparison[
        "baseline_squared_error"
    ].mean()
)

# Compare results
print("7-DAY FORECAST MODEL COMPARISON")
print("--------------------------------")
print(f"Random Forest MAE: {overall_mae:.2f}")
print(f"Naive Baseline MAE: {baseline_mae:.2f}")
print()
print(f"Random Forest RMSE: {overall_rmse:.2f}")
print(f"Naive Baseline RMSE: {baseline_rmse:.2f}")

improvement = (
    (baseline_mae - overall_mae)
    / baseline_mae
) * 100 if baseline_mae > 0 else float("nan")

print(f"\nMAE Improvement: {improvement:.2f}%")

7-DAY FORECAST MODEL COMPARISON
--------------------------------
Random Forest MAE: 12.69
Naive Baseline MAE: 18.54

Random Forest RMSE: 20.91
Naive Baseline RMSE: 26.67

MAE Improvement: 31.55%


In [33]:
# Inspect the original feature-engineering code
import json
from pathlib import Path

notebook_path = Path("01_data_exploration.ipynb")

if not notebook_path.exists():
    notebook_path = Path("notebooks/01_data_exploration.ipynb")

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "shift(",
    "rolling(",
    "lag_1",
    "lag_7",
    "lag_28"
]

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if cell.get("cell_type") == "code" and any(
        word in source for word in keywords
    ):
        print(f"\n{'=' * 15} CELL {i} {'=' * 15}")
        print(source[:4000])


=============== CELL 13 ===============
plt.figure(figsize=(14, 6))

for _, product in top_5_ca1.iterrows():
    daily_sales = product[day_columns].astype(float)

    rolling_sales = daily_sales.rolling(
        window=30,
        min_periods=1
    ).mean()

    plt.plot(
        range(1, 1914),
        rolling_sales,
        label=product["item_id"]
    )

plt.title("30-Day Moving Average - Top 5 Products in CA_1")
plt.xlabel("Day")
plt.ylabel("Average Units Sold")
plt.legend()
plt.grid(True)
plt.show()

=============== CELL 18 ===============
# Sort records by product and date
top_5_merged = top_5_merged.sort_values(
    ["item_id", "store_id", "date"]
).copy()

# Create lag features for each product separately
for lag in [1, 7, 28]:
    top_5_merged[f"lag_{lag}"] = (
        top_5_merged.groupby(["item_id", "store_id"])["sales"]
        .shift(lag)
    )

# Display one product to verify the lag values
display(
    top_5_merged[
        top_5_merged["item_id"] == "FOODS_3_090"
    ]

In [34]:
# Select non-overlapping 7-day test windows
rolling_starts = pd.date_range(
    start="2015-07-17",
    end="2016-04-17",
    freq="28D"
)

rolling_windows = pd.DataFrame({
    "start_date": rolling_starts,
    "end_date": rolling_starts + pd.Timedelta(days=6)
})

# Verify every window is inside the held-out test period
test_start = pd.Timestamp("2015-07-17")
test_end = pd.Timestamp("2016-04-24")

rolling_windows["valid"] = (
    (rolling_windows["start_date"] >= test_start)
    & (rolling_windows["end_date"] <= test_end)
)

print("Total backtesting windows:", len(rolling_windows))
print("All windows valid:", rolling_windows["valid"].all())

display(rolling_windows)

Total backtesting windows: 10
All windows valid: True


,start_date,end_date,valid
0,2015-07-17,2015-07-23,True
1,2015-08-14,2015-08-20,True
2,2015-09-11,2015-09-17,True
3,2015-10-09,2015-10-15,True
4,2015-11-06,2015-11-12,True
5,2015-12-04,2015-12-10,True
6,2016-01-01,2016-01-07,True
7,2016-01-29,2016-02-04,True
8,2016-02-26,2016-03-03,True
9,2016-03-25,2016-03-31,True


In [35]:
def run_rolling_backtest(start_date):

    global backtest_history
    global backtest_dates
    global backtest_calendar
    global backtest_prices

    start_date = pd.Timestamp(start_date)

    # Seven-day forecasting window
    dates = pd.date_range(
        start=start_date,
        periods=7,
        freq="D"
    )

    # Historical sales available before forecast start
    history = {}

    for item in selected_products:

        product_data = historical_df[
            (historical_df["item_id"] == item)
            & (historical_df["date"] < start_date)
        ].sort_values("date")

        history[item] = (
            product_data["sales"].astype(float).tolist()
        )

    # Calendar for forecast dates
    calendar = pd.read_csv(calendar_path)
    calendar["date"] = pd.to_datetime(calendar["date"])

    calendar = calendar[
        calendar["date"].isin(dates)
    ].copy()

    # Prices for the relevant weeks
    weeks = calendar["wm_yr_wk"].unique()

    prices = prices_df[
        (prices_df["store_id"] == "CA_1")
        & (prices_df["item_id"].isin(selected_products))
        & (prices_df["wm_yr_wk"].isin(weeks))
    ].copy()

    # Verify all required inputs are available
    assert len(calendar) == 7
    assert all(len(h) >= 28 for h in history.values())
    assert len(prices) == len(selected_products) * len(weeks)
    assert prices["sell_price"].notna().all()

    # Preserve existing backtest inputs
    old_inputs = (
        backtest_history,
        backtest_dates,
        backtest_calendar,
        backtest_prices
    )

    try:
        backtest_history = history
        backtest_dates = dates
        backtest_calendar = calendar
        backtest_prices = prices

        predictions = pd.concat(
            [
                forecast_backtest(item, days=7)
                for item in selected_products
            ],
            ignore_index=True
        )

    finally:
        (
            backtest_history,
            backtest_dates,
            backtest_calendar,
            backtest_prices
        ) = old_inputs

    predictions["window_start"] = start_date

    return predictions


print("Rolling backtest function created successfully!")

Rolling backtest function created successfully!


In [36]:
# Run recursive forecasting for all 10 test windows
rolling_prediction_list = []

for start_date in rolling_windows["start_date"]:

    print(f"Forecasting window: {start_date.date()}")

    window_predictions = run_rolling_backtest(
        start_date
    )

    rolling_prediction_list.append(
        window_predictions
    )

# Combine all predictions
rolling_predictions = pd.concat(
    rolling_prediction_list,
    ignore_index=True
)

# Verify results
print("\nROLLING BACKTEST RESULTS")
print("------------------------")
print("Total predictions:", len(rolling_predictions))
print("Total windows:",
      rolling_predictions["window_start"].nunique())
print("Total products:",
      rolling_predictions["item_id"].nunique())

print("Missing predictions:",
      rolling_predictions["predicted_sales"].isna().sum())

display(rolling_predictions.head(10))

Forecasting window: 2015-07-17
Forecasting window: 2015-08-14
Forecasting window: 2015-09-11
Forecasting window: 2015-10-09
Forecasting window: 2015-11-06
Forecasting window: 2015-12-04
Forecasting window: 2016-01-01
Forecasting window: 2016-01-29
Forecasting window: 2016-02-26
Forecasting window: 2016-03-25

ROLLING BACKTEST RESULTS
------------------------
Total predictions: 350
Total windows: 10
Total products: 5
Missing predictions: 0


,item_id,date,predicted_sales,window_start
0,FOODS_3_090,2015-07-17,82.327760,2015-07-17
1,FOODS_3_090,2015-07-18,92.180783,2015-07-17
2,FOODS_3_090,2015-07-19,91.678964,2015-07-17
3,FOODS_3_090,2015-07-20,55.345552,2015-07-17
4,FOODS_3_090,2015-07-21,53.262669,2015-07-17
5,FOODS_3_090,2015-07-22,52.727912,2015-07-17
6,FOODS_3_090,2015-07-23,55.273953,2015-07-17
7,FOODS_3_120,2015-07-17,40.388753,2015-07-17
8,FOODS_3_120,2015-07-18,55.361086,2015-07-17
9,FOODS_3_120,2015-07-19,60.290023,2015-07-17


In [37]:
# Get actual sales for all rolling forecast dates
actual_sales = historical_df[
    ["item_id", "date", "sales"]
].copy()

# Match forecasts with actual sales
rolling_results = rolling_predictions.merge(
    actual_sales,
    on=["item_id", "date"],
    how="left",
    validate="many_to_one"
)

# Verify all predictions have actual sales
assert len(rolling_results) == 350
assert rolling_results["sales"].notna().all()

# Calculate forecast errors
rolling_results["absolute_error"] = (
    rolling_results["sales"]
    - rolling_results["predicted_sales"]
).abs()

rolling_results["squared_error"] = (
    rolling_results["sales"]
    - rolling_results["predicted_sales"]
) ** 2

# Overall metrics
rolling_mae = rolling_results["absolute_error"].mean()

rolling_rmse = np.sqrt(
    rolling_results["squared_error"].mean()
)

print("10-WINDOW ROLLING BACKTEST")
print("--------------------------")
print("Total predictions:", len(rolling_results))
print("MAE:", round(rolling_mae, 2))
print("RMSE:", round(rolling_rmse, 2))

# Product-level performance
rolling_product_metrics = rolling_results.groupby(
    "item_id"
).agg(
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean")
)

rolling_product_metrics["RMSE"] = np.sqrt(
    rolling_product_metrics["MSE"]
)

rolling_product_metrics = rolling_product_metrics[
    ["MAE", "RMSE"]
].round(2)

print("\nProduct-Level Performance:")
display(rolling_product_metrics)

10-WINDOW ROLLING BACKTEST
--------------------------
Total predictions: 350
MAE: 11.83
RMSE: 16.92

Product-Level Performance:


,MAE,RMSE
item_id,,
FOODS_3_090,18.45,23.62
FOODS_3_120,18.52,24.34
FOODS_3_252,7.94,10.49
FOODS_3_586,7.50,9.74
FOODS_3_714,6.75,8.76


In [38]:
# Build seasonal naive predictions for all 10 windows
baseline_rows = []

for start_date in rolling_windows["start_date"]:

    for item in selected_products:

        history = historical_df[
            (historical_df["item_id"] == item)
            & (historical_df["date"] < start_date)
        ].sort_values("date")

        previous_week = history["sales"].tail(7).tolist()

        for i in range(7):

            baseline_rows.append({
                "window_start": start_date,
                "item_id": item,
                "date": start_date + pd.Timedelta(days=i),
                "baseline_sales": previous_week[i]
            })

rolling_baseline = pd.DataFrame(baseline_rows)

# Combine baseline with model results
comparison = rolling_results.merge(
    rolling_baseline,
    on=["window_start", "item_id", "date"],
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 350

# Calculate baseline errors
baseline_mae_rolling = (
    comparison["sales"] - comparison["baseline_sales"]
).abs().mean()

baseline_rmse_rolling = np.sqrt(
    (
        (comparison["sales"] - comparison["baseline_sales"]) ** 2
    ).mean()
)

# Percentage improvement
mae_improvement_rolling = (
    (baseline_mae_rolling - rolling_mae)
    / baseline_mae_rolling
) * 100

print("10-WINDOW MODEL COMPARISON")
print("--------------------------")
print(f"Random Forest MAE: {rolling_mae:.2f}")
print(f"Seasonal Naive MAE: {baseline_mae_rolling:.2f}")

print()
print(f"Random Forest RMSE: {rolling_rmse:.2f}")
print(f"Seasonal Naive RMSE: {baseline_rmse_rolling:.2f}")

print()
print(f"MAE Improvement: {mae_improvement_rolling:.2f}%")

10-WINDOW MODEL COMPARISON
--------------------------
Random Forest MAE: 11.83
Seasonal Naive MAE: 14.56

Random Forest RMSE: 16.92
Seasonal Naive RMSE: 20.87

MAE Improvement: 18.73%


In [39]:
# Calculate forecast horizon: Day 1 through Day 7
comparison["horizon"] = (
    comparison["date"] - comparison["window_start"]
).dt.days + 1

# Calculate baseline absolute errors
comparison["baseline_absolute_error"] = (
    comparison["sales"] - comparison["baseline_sales"]
).abs()

# Compare accuracy at each forecast horizon
horizon_metrics = comparison.groupby("horizon").agg(
    RF_MAE=("absolute_error", "mean"),
    Baseline_MAE=("baseline_absolute_error", "mean"),
    Predictions=("sales", "count")
)

horizon_metrics["Improvement_%"] = (
    (horizon_metrics["Baseline_MAE"] - horizon_metrics["RF_MAE"])
    / horizon_metrics["Baseline_MAE"]
) * 100

print("FORECAST ACCURACY BY HORIZON")
print("----------------------------")

display(horizon_metrics.round(2))

FORECAST ACCURACY BY HORIZON
----------------------------


,RF_MAE,Baseline_MAE,Predictions,Improvement_%
horizon,,,,
1,9.06,14.76,50,38.63
2,15.51,15.88,50,2.34
3,16.28,16.04,50,-1.47
4,14.21,18.26,50,22.19
5,11.09,12.50,50,11.30
6,8.23,13.22,50,37.73
7,8.46,11.26,50,24.88


In [40]:
# Create evaluation output directory
evaluation_dir = PROJECT_DIR / "data" / "processed"
evaluation_dir.mkdir(parents=True, exist_ok=True)

# Save detailed predictions and baseline comparison
comparison.to_csv(
    evaluation_dir / "rolling_backtest_results.csv",
    index=False
)

# Save product-level performance
rolling_product_metrics.to_csv(
    evaluation_dir / "product_forecast_metrics.csv"
)

# Save forecast horizon performance
horizon_metrics.to_csv(
    evaluation_dir / "forecast_horizon_metrics.csv"
)

print("Evaluation files saved successfully!")
print("\nFiles created:")

for filename in [
    "rolling_backtest_results.csv",
    "product_forecast_metrics.csv",
    "forecast_horizon_metrics.csv"
]:
    filepath = evaluation_dir / filename

    print(
        f"{filename} | "
        f"Size: {filepath.stat().st_size:,} bytes"
    )

Evaluation files saved successfully!

Files created:
rolling_backtest_results.csv | Size: 35,309 bytes
product_forecast_metrics.csv | Size: 137 bytes
forecast_horizon_metrics.csv | Size: 395 bytes
